## **Measuring the Impact of Telemarketing Strategies on a Portuguese Bank**

**Introduction**

As competition in the banking industry grows and new investment options emerge, firms are turning to marketing as a way to stand out. In this notebook, will focus on analyzing the impact of telemarketing campaigns have been for a Portuguese bank. The bank has been running telemarketing campaigns to promote term deposits, which are a type of investment product. We will use hypothesis testing to determine if the telemarketing campaigns have been effective in increasing the number of term deposits.

## **Hypothesis Testing**

__Will perform hypothesis testing on bank-full.csv file. The dataset contains information of clients contacted during marketing campaigns of a Protuguese bank - contains 45,211 clients with 16 features. Features describes each client's loan information, job, demography, education level, transaction history, and so on. with a binary target variable indicating whether the client subscribed to a term deposit. The data spans from May 2008 to November 2010.__

**Brief description of Columns :**

__Bank Client Data Description:__

- **age :** Age of the client(numeric)
- **job :** type of job (categorical): 'admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown'
- **marital:** marital status (categorical): 'divorced', 'married', 'single', 'unknown'; note: 'divorced' means divorced or widowed
- **education:** (categorical): Level of Education ('primary', 'secondary', 'tertiary' , 'unknown')
- **default:** Whether the client has credit in default (categorical): 'no', 'yes', 'unknown'
- **balance:** customer balance (numeric)
- **housing:** Whether the client has a housing loan  (categorical): 'no', 'yes', 'unknown'
- **loan:** Whether the client has a personal loan (categorical): 'no', 'yes', 'unknown'

__Last Contact Data Description__
- **contact:** last contact communication type (categorical): 'cellular', 'telephone'
- **month:** last contact month of year (categorical): 'jan', 'feb', 'mar', ..., 'nov', 'dec'
- **day:** last contact day of the week (numerical): 1-5
- **duration:** last contact duration, in seconds (numeric).
- **campaign:** number of contacts, including the last contact, performed during this campaign and for this client (numeric )
- **pdays:** number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 or -1 means client was not previously contacted)
- **previous:** number of contacts performed before this campaign and for this client (numeric)
- **poutcome:** outcome of the previous marketing campaign (categorical): 'failure', 'nonexistent', 'success'

__NOTE__ : Duration (last contact duration, in seconds (numeric). Important note:  this attribute highly affects the output target (e.g., if duration=0 then y="no"). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.)


In [ ]:
## Import commonly used libraries
import pandas as pd 
import numpy as np  
import sidetable
import sklearn
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
from itertools import combinations


In [2]:
## Dispaly Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Read the data
files = []
data_path = Path.cwd().parent.joinpath('data', 'raw')
for file in data_path.glob('*'):
    files.append(file.name)

print(files)

['.gitkeep', 'bank-additional-full.csv', 'bank-additional-names.txt', 'bank-full.csv', 'info.txt']


In [13]:
f = open(data_path.joinpath('info.txt'), 'rt')
for line in f.readlines():
    print(line.strip())

Bank Telemarketing Dataset consists of two datasets namely bank-full and bank-additional-full.


In [16]:
df = pd.read_csv(data_path.joinpath('bank-full.csv'), sep=';')
df.head(1)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no


In [7]:
df.shape

(45211, 17)

In [19]:
# Remove whitespaces, ., spaces and lowercase column names
df.columns = df.columns.str.strip().str.replace(".", " ").str.replace(" ", "_").str.lower()
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


# __Formatting and Consistency Checks__

In [21]:
dtype_df = pd.DataFrame({
    'dtype': df.dtypes,
    'nunique': df.nunique(),
    'unique' : [df[col].unique() for col in df.columns]
}, index=df.columns)
dtype_df

,dtype,nunique,unique
age,int64,77,"[58, 44, 33, 47, 35, 28, 42, 43, 41, 29, 53, 5..."
job,object,12,"[management, technician, entrepreneur, blue-co..."
marital,object,3,"[married, single, divorced]"
education,object,4,"[tertiary, secondary, unknown, primary]"
default,object,2,"[no, yes]"
balance,int64,7168,"[2143, 29, 2, 1506, 1, 231, 447, 121, 593, 270..."
housing,object,2,"[yes, no]"
loan,object,2,"[no, yes]"
contact,object,3,"[unknown, cellular, telephone]"
day,int64,31,"[5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 19, 20, 21..."
